In [0]:
# Python utilities used for JSON handling and batch timestamps.
import json
from datetime import datetime, timezone
from pathlib import Path

# Spark functions and types used to construct the Bronze dataframe.
from pyspark.sql import functions as F
from pyspark.sql.types import (
    StructType,
    StructField,
    StringType,
    IntegerType,
    TimestampType,
)


# Stable identifier included in every lineage record.
PROJECT_NAME = "trichy_east_election_pipeline"


# Unity Catalog destination for raw polling-booth records.
CATALOG = "workspace"
BRONZE_SCHEMA = "bronze"


# Root Volume containing the 2016, 2021 and 2026 directories.
BASE_VOLUME_PATH = "/Volumes/workspace/elections/trichy_east"


# Stable constituency metadata added even when a page header is unreadable.
CONSTITUENCY_NO = "141"
CONSTITUENCY_NAME = "Tiruchirappalli East"


# Restrict this notebook to polling-booth reference sources.
SOURCE_TYPE = "polling_booths"


# Unique run identifier for traceability and idempotent writes.
BATCH_ID = datetime.now(timezone.utc).strftime(
    "polling_booths_%Y%m%dT%H%M%SZ"
)


# Raw Bronze target table.
BRONZE_TABLE = (
    f"{CATALOG}.{BRONZE_SCHEMA}."
    "trichy_east_polling_booths_raw"
)


# Git-managed source configuration.
MANIFEST_PATH = Path("config/source_manifest.json")


print("Project:", PROJECT_NAME)
print("Source type:", SOURCE_TYPE)
print("Batch ID:", BATCH_ID)
print("Target table:", BRONZE_TABLE)
print("Manifest:", MANIFEST_PATH)

In [0]:
# Stop early if the Git-managed configuration is unavailable.
if not MANIFEST_PATH.exists():
    raise FileNotFoundError(
        f"Source manifest was not found at {MANIFEST_PATH}. "
        "Pull the latest Git changes in Databricks."
    )


# Load the configuration maintained in GitHub.
with MANIFEST_PATH.open("r") as manifest_file:
    source_manifest = json.load(manifest_file)


# Protect against reading from an unintended Volume.
assert (
    source_manifest["databricks_base_volume"]
    == BASE_VOLUME_PATH
), "The notebook and manifest Volume paths do not match."


# Select only polling-booth reference files.
polling_booth_source_rows = []

for source in source_manifest["sources"]:
    if source["source_type"] != SOURCE_TYPE:
        continue

    source_file_path = (
        f"{BASE_VOLUME_PATH.rstrip('/')}/"
        f"{source['relative_path'].lstrip('/')}"
    )

    polling_booth_source_rows.append(
        {
            "project_name": PROJECT_NAME,
            "constituency_no": CONSTITUENCY_NO,
            "constituency_name": CONSTITUENCY_NAME,
            "election_year": int(source["election_year"]),
            "source_type": source["source_type"],
            "source_format": source["format"],
            "source_file_name": source["file_name"],
            "source_file_path": source_file_path,
            "configured_extraction_strategy": source[
                "extraction_strategy"
            ],
            "batch_id": BATCH_ID,
        }
    )


# Convert configuration records into a Spark control dataframe.
polling_booth_source_df = spark.createDataFrame(
    polling_booth_source_rows
)


display(
    polling_booth_source_df.orderBy(
        "election_year"
    )
)


print(
    "Configured polling-booth files:",
    polling_booth_source_df.count()
)

In [0]:
# Python utility used to inspect files mounted under /Volumes.
import os


polling_booth_file_check_rows = []


for source in polling_booth_source_rows:
    source_file_path = source["source_file_path"]

    try:
        file_exists = os.path.isfile(source_file_path)

        if file_exists:
            file_size_bytes = int(
                os.path.getsize(source_file_path)
            )
            file_status = "AVAILABLE"
            validation_error = None
        else:
            file_size_bytes = None
            file_status = "MISSING"
            validation_error = (
                "Configured polling-booth file was not found."
            )

    except Exception as error:
        file_exists = False
        file_size_bytes = None
        file_status = "ERROR"
        validation_error = str(error)

    polling_booth_file_check_rows.append(
        {
            **source,
            "file_exists": file_exists,
            "file_size_bytes": file_size_bytes,
            "file_status": file_status,
            "validation_error": validation_error,
        }
    )


# Explicit schema prevents errors when all validation errors are null.
polling_booth_file_check_schema = StructType([
    StructField("project_name", StringType(), False),
    StructField("constituency_no", StringType(), False),
    StructField("constituency_name", StringType(), False),
    StructField("election_year", IntegerType(), False),
    StructField("source_type", StringType(), False),
    StructField("source_format", StringType(), False),
    StructField("source_file_name", StringType(), False),
    StructField("source_file_path", StringType(), False),
    StructField(
        "configured_extraction_strategy",
        StringType(),
        False
    ),
    StructField("batch_id", StringType(), False),
    StructField("file_exists", StringType(), False),
    StructField("file_size_bytes", StringType(), True),
    StructField("file_status", StringType(), False),
    StructField("validation_error", StringType(), True),
])


polling_booth_file_check_df = spark.createDataFrame(
    polling_booth_file_check_rows,
    schema=polling_booth_file_check_schema
)


display(
    polling_booth_file_check_df.select(
        "election_year",
        "source_file_name",
        "source_file_path",
        "file_size_bytes",
        "file_status",
        "validation_error"
    ).orderBy("election_year")
)


unavailable_file_count = (
    polling_booth_file_check_df
    .filter(F.col("file_status") != "AVAILABLE")
    .count()
)


print(
    "Polling-booth files:",
    polling_booth_file_check_df.count()
)
print("Unavailable files:", unavailable_file_count)


if unavailable_file_count > 0:
    raise ValueError(
        f"{unavailable_file_count} polling-booth file(s) are unavailable."
    )

In [0]:
# Pin the PDF library version so notebook runs are reproducible.
%pip install pypdf==6.10.0

In [0]:
# Import the PDF reader after installation.
import pypdf
from pypdf import PdfReader


print("pypdf version:", pypdf.__version__)
print("Polling-booth PDF extraction dependency: READY")

In [0]:
# Preserve raw station details without address normalization or joining.
POLLING_BOOTHS_BRONZE_SCHEMA = StructType([
    # Source lineage
    StructField("project_name", StringType(), False),
    StructField("source_file_name", StringType(), False),
    StructField("source_file_path", StringType(), False),
    StructField("source_type", StringType(), False),

    # Stable election metadata
    StructField("constituency_name", StringType(), False),
    StructField("constituency_no", StringType(), False),
    StructField("election_year", IntegerType(), False),

    # Position inside the PDF
    StructField("page_no", IntegerType(), True),
    StructField("row_no", IntegerType(), True),

    # Raw station fields when safely observable
    StructField("serial_no_raw", StringType(), True),
    StructField("polling_station_no_raw", StringType(), True),
    StructField("building_name_raw", StringType(), True),
    StructField("polling_area_raw", StringType(), True),
    StructField("voter_type_raw", StringType(), True),

    # Complete raw extraction
    StructField("raw_row_text", StringType(), True),
    StructField("raw_fields_json", StringType(), True),

    # Extraction diagnostics
    StructField(
        "extraction_metadata_json",
        StringType(),
        True
    ),

    # Parsing outcome
    StructField("parse_status", StringType(), False),
    StructField("parse_error", StringType(), True),
    StructField("extraction_method", StringType(), False),

    # Run lineage
    StructField("batch_id", StringType(), False),
    StructField("ingestion_timestamp", TimestampType(), True),
    StructField("extraction_timestamp", TimestampType(), True),
])


print(
    "Polling-booth Bronze columns:",
    len(POLLING_BOOTHS_BRONZE_SCHEMA.fields)
)
print("Target table:", BRONZE_TABLE)

In [0]:
def clean_polling_booth_line(raw_line):
    """
    Apply technical text cleanup only.

    It removes null characters and repeated whitespace but does not
    standardize addresses, station names or polling areas.
    """
    if raw_line is None:
        return ""

    return " ".join(
        raw_line.replace("\x00", "").split()
    ).strip()


def build_polling_booth_record(
    source,
    page_no,
    row_no,
    raw_row_text,
    extraction_method,
    parse_status="SUCCESS",
    parse_error=None,
    raw_fields=None,
    extraction_metadata=None,
):
    """
    Create one record matching the stable polling-booth Bronze schema.
    """
    return {
        "project_name": source["project_name"],
        "source_file_name": source["source_file_name"],
        "source_file_path": source["source_file_path"],
        "source_type": source["source_type"],
        "constituency_name": source["constituency_name"],
        "constituency_no": source["constituency_no"],
        "election_year": source["election_year"],
        "page_no": page_no,
        "row_no": row_no,

        # These remain null until the source text can be interpreted safely.
        "serial_no_raw": None,
        "polling_station_no_raw": None,
        "building_name_raw": None,
        "polling_area_raw": None,
        "voter_type_raw": None,

        "raw_row_text": raw_row_text,
        "raw_fields_json": (
            json.dumps(raw_fields, ensure_ascii=False)
            if raw_fields is not None
            else None
        ),
        "extraction_metadata_json": (
            json.dumps(
                extraction_metadata,
                ensure_ascii=False
            )
            if extraction_metadata is not None
            else None
        ),
        "parse_status": parse_status,
        "parse_error": parse_error,
        "extraction_method": extraction_method,
        "batch_id": source["batch_id"],
        "ingestion_timestamp": None,
        "extraction_timestamp": datetime.now(
            timezone.utc
        ),
    }


def extract_polling_booth_pdf(source):
    """
    Extract a polling-station PDF page by page using pypdf layout mode.

    Empty or failed pages still create failure records so that source
    coverage remains visible.
    """
    extracted_records = []

    try:
        pdf_reader = PdfReader(
            source["source_file_path"]
        )

        total_pages = len(pdf_reader.pages)

        for page_index, pdf_page in enumerate(
            pdf_reader.pages,
            start=1
        ):
            try:
                page_text = pdf_page.extract_text(
                    extraction_mode="layout"
                ) or ""

                extracted_lines = [
                    clean_polling_booth_line(line)
                    for line in page_text.splitlines()
                    if clean_polling_booth_line(line)
                ]

                if not extracted_lines:
                    extracted_records.append(
                        build_polling_booth_record(
                            source=source,
                            page_no=page_index,
                            row_no=None,
                            raw_row_text=None,
                            extraction_method="pypdf_layout_text",
                            parse_status="FAILED",
                            parse_error=(
                                "No text extracted from polling-booth page."
                            ),
                            raw_fields=None,
                            extraction_metadata={
                                "parser": "pypdf",
                                "extraction_mode": "layout",
                                "pdf_page": page_index,
                                "pdf_page_count": total_pages,
                                "extracted_line_count": 0,
                            },
                        )
                    )

                    continue

                for row_index, extracted_line in enumerate(
                    extracted_lines,
                    start=1
                ):
                    extracted_records.append(
                        build_polling_booth_record(
                            source=source,
                            page_no=page_index,
                            row_no=row_index,
                            raw_row_text=extracted_line,
                            extraction_method="pypdf_layout_text",
                            raw_fields={
                                "line_1": extracted_line
                            },
                            extraction_metadata={
                                "parser": "pypdf",
                                "extraction_mode": "layout",
                                "pdf_page": page_index,
                                "pdf_page_count": total_pages,
                                "extracted_line_count": len(
                                    extracted_lines
                                ),
                            },
                        )
                    )

            except Exception as page_error:
                extracted_records.append(
                    build_polling_booth_record(
                        source=source,
                        page_no=page_index,
                        row_no=None,
                        raw_row_text=None,
                        extraction_method="pypdf_layout_text",
                        parse_status="FAILED",
                        parse_error=str(page_error),
                        extraction_metadata={
                            "parser": "pypdf",
                            "pdf_page": page_index,
                        },
                    )
                )

    except Exception as file_error:
        extracted_records.append(
            build_polling_booth_record(
                source=source,
                page_no=None,
                row_no=None,
                raw_row_text=None,
                extraction_method="pypdf_layout_text",
                parse_status="FAILED",
                parse_error=str(file_error),
                extraction_metadata={
                    "parser": "pypdf"
                },
            )
        )

    return extracted_records


print("Raw polling-booth PDF extractor: READY")

In [0]:
# Collect raw records from all configured years.
all_polling_booth_records = []


for source in sorted(
    polling_booth_source_rows,
    key=lambda row: row["election_year"]
):
    print(
        f"Extracting {source['election_year']}: "
        f"{source['source_file_name']}"
    )

    source_records = extract_polling_booth_pdf(
        source
    )

    print(
        "Records extracted:",
        len(source_records)
    )

    all_polling_booth_records.extend(
        source_records
    )


# Build the complete Bronze dataframe with explicit types.
polling_booths_bronze_df = spark.createDataFrame(
    all_polling_booth_records,
    schema=POLLING_BOOTHS_BRONZE_SCHEMA
).withColumn(
    "ingestion_timestamp",
    F.current_timestamp()
)


# Show bounded extraction coverage by year and status.
display(
    polling_booths_bronze_df
    .groupBy(
        "election_year",
        "source_file_name",
        "parse_status",
        "extraction_method"
    )
    .count()
    .orderBy(
        "election_year",
        "parse_status"
    )
)


print(
    "Total polling-booth Bronze records:",
    polling_booths_bronze_df.count()
)

In [0]:
# PyMuPDF handles rotated or unusually encoded PDF text that pypdf
# layout extraction cannot read reliably.
%pip install PyMuPDF==1.26.7

In [0]:
# PyMuPDF is imported using the module name "fitz".
import fitz

print("PyMuPDF version:", fitz.VersionBind)
print("Polling-booth fallback extractor: READY")

In [0]:
import fitz


def extract_polling_booth_pdf_with_pymupdf(source):
    """
    Extract raw text lines using PyMuPDF.

    This is the fallback for PDFs where pypdf cannot correctly
    handle rotated or unusually positioned text.
    """
    records = []
    document = None

    try:
        document = fitz.open(source["source_file_path"])

        for page_index in range(document.page_count):
            page_no = page_index + 1
            page = document.load_page(page_index)

            # Preserve the text returned by PyMuPDF without
            # applying polling-booth business rules.
            page_text = page.get_text("text") or ""

            raw_lines = [
                clean_polling_booth_line(line)
                for line in page_text.splitlines()
            ]

            # Remove only blank lines.
            extracted_lines = [
                line for line in raw_lines if line
            ]

            if not extracted_lines:
                records.append(
                    build_polling_booth_record(
                        source=source,
                        page_no=page_no,
                        row_no=None,
                        raw_row_text=None,
                        extraction_method="pymupdf_text",
                        parse_status="FAILED",
                        parse_error="No text extracted from page.",
                        raw_fields=None,
                        extraction_metadata={
                            "fallback_reason": "pypdf_rotated_text",
                            "page_index_zero_based": page_index
                        }
                    )
                )
                continue

            for row_no, line in enumerate(
                extracted_lines,
                start=1
            ):
                records.append(
                    build_polling_booth_record(
                        source=source,
                        page_no=page_no,
                        row_no=row_no,
                        raw_row_text=line,
                        extraction_method="pymupdf_text",
                        parse_status="SUCCESS",
                        parse_error=None,
                        raw_fields={
                            "line_1": line
                        },
                        extraction_metadata={
                            "fallback_reason": "pypdf_rotated_text",
                            "page_index_zero_based": page_index
                        }
                    )
                )

    except Exception as error:
        records.append(
            build_polling_booth_record(
                source=source,
                page_no=None,
                row_no=None,
                raw_row_text=None,
                extraction_method="pymupdf_text",
                parse_status="FAILED",
                parse_error=str(error),
                raw_fields=None,
                extraction_metadata={
                    "failure_scope": "file"
                }
            )
        )

    finally:
        if document is not None:
            document.close()

    return records


print("PyMuPDF polling-booth fallback function: READY")

In [0]:
# Find files that already produced successful records.
primary_success_paths = {
    row["source_file_path"]
    for row in (
        polling_booths_bronze_df
        .filter(F.col("parse_status") == "SUCCESS")
        .select("source_file_path")
        .distinct()
        .collect()
    )
}


# Keep successful primary extraction files, such as 2026.
primary_records_df = polling_booths_bronze_df.filter(
    F.col("source_file_path").isin(
        list(primary_success_paths)
    )
)


# Run PyMuPDF only for files where pypdf produced no successful records.
fallback_records = []

for source in polling_booth_source_rows:
    if source["source_file_path"] not in primary_success_paths:
        print(
            f"Running fallback for {source['election_year']}: "
            f"{source['source_file_name']}"
        )

        source_records = (
            extract_polling_booth_pdf_with_pymupdf(source)
        )

        print("Fallback records:", len(source_records))
        fallback_records.extend(source_records)


# Convert fallback records to the same stable Bronze schema.
fallback_records_df = (
    spark.createDataFrame(
        fallback_records,
        schema=POLLING_BOOTHS_BRONZE_SCHEMA
    )
    .withColumn(
        "ingestion_timestamp",
        F.current_timestamp()
    )
)


# Combine primary and fallback extraction results.
final_polling_booths_bronze_df = (
    primary_records_df
    .unionByName(fallback_records_df)
)


# Confirm which extraction method succeeded for each year.
display(
    final_polling_booths_bronze_df
    .groupBy(
        "election_year",
        "source_file_name",
        "parse_status",
        "extraction_method"
    )
    .count()
    .orderBy(
        "election_year",
        "parse_status"
    )
)

print(
    "Final polling-booth Bronze records:",
    final_polling_booths_bronze_df.count()
)

In [0]:
from pyspark.sql.window import Window


# Check that every configured file produced successful records.
successful_source_count = (
    final_polling_booths_bronze_df
    .filter(F.col("parse_status") == "SUCCESS")
    .select("source_file_path")
    .distinct()
    .count()
)

print("Expected source files:", len(polling_booth_source_rows))
print("Successful source files:", successful_source_count)

assert successful_source_count == len(
    polling_booth_source_rows
), "At least one polling-booth source has no successful records."


# A source page and row number should identify one raw record.
duplicate_coordinates_df = (
    final_polling_booths_bronze_df
    .filter(F.col("parse_status") == "SUCCESS")
    .groupBy(
        "source_file_path",
        "page_no",
        "row_no"
    )
    .count()
    .filter(F.col("count") > 1)
)

duplicate_coordinate_count = duplicate_coordinates_df.count()

print(
    "Duplicate source coordinates:",
    duplicate_coordinate_count
)

assert duplicate_coordinate_count == 0, (
    "Duplicate source page/row coordinates were found."
)


# Display only five raw records from each year for visual inspection.
sample_window = Window.partitionBy(
    "election_year"
).orderBy(
    "page_no",
    "row_no"
)

polling_booth_sample_df = (
    final_polling_booths_bronze_df
    .filter(F.col("parse_status") == "SUCCESS")
    .withColumn(
        "sample_row_number",
        F.row_number().over(sample_window)
    )
    .filter(F.col("sample_row_number") <= 5)
    .select(
        "election_year",
        "source_file_name",
        "page_no",
        "row_no",
        "raw_row_text",
        "extraction_method"
    )
    .orderBy(
        "election_year",
        "page_no",
        "row_no"
    )
)

display(polling_booth_sample_df)

print("Polling-booth Bronze QA: PASS")

In [0]:
POLLING_BOOTHS_BRONZE_TABLE = (
    "workspace.bronze.trichy_east_polling_booths_raw"
)

spark.sql(
    "CREATE SCHEMA IF NOT EXISTS workspace.bronze"
)

if spark.catalog.tableExists(
    POLLING_BOOTHS_BRONZE_TABLE
):
    spark.sql(
        f"""
        DELETE FROM {POLLING_BOOTHS_BRONZE_TABLE}
        WHERE batch_id = '{BATCH_ID}'
        """
    )
    write_mode = "append"
else:
    write_mode = "overwrite"


(
    final_polling_booths_bronze_df
    .write
    .format("delta")
    .mode(write_mode)
    .saveAsTable(POLLING_BOOTHS_BRONZE_TABLE)
)

print(
    "Rows written:",
    final_polling_booths_bronze_df.count()
)
print("Target table:", POLLING_BOOTHS_BRONZE_TABLE)
print("Batch ID:", BATCH_ID)

In [0]:
# Read the records back from Delta to verify the completed write.
written_polling_booths_df = (
    spark.table(POLLING_BOOTHS_BRONZE_TABLE)
    .filter(F.col("batch_id") == BATCH_ID)
)


written_row_count = written_polling_booths_df.count()

expected_row_count = (
    final_polling_booths_bronze_df.count()
)

written_source_count = (
    written_polling_booths_df
    .select("source_file_path")
    .distinct()
    .count()
)

successful_source_count = (
    written_polling_booths_df
    .filter(F.col("parse_status") == "SUCCESS")
    .select("source_file_path")
    .distinct()
    .count()
)


print("Expected rows:", expected_row_count)
print("Rows read from Delta:", written_row_count)
print("Source files written:", written_source_count)
print("Sources with successful extraction:", successful_source_count)

assert written_row_count == expected_row_count
assert written_source_count == 3
assert successful_source_count == 3

display(
    written_polling_booths_df
    .groupBy(
        "election_year",
        "parse_status",
        "extraction_method"
    )
    .count()
    .orderBy("election_year")
)

print("Polling-booth Bronze write validation: PASS")

In [0]:
# Read the records back from Delta to verify the completed write.
written_polling_booths_df = (
    spark.table(POLLING_BOOTHS_BRONZE_TABLE)
    .filter(F.col("batch_id") == BATCH_ID)
)


written_row_count = written_polling_booths_df.count()

expected_row_count = (
    final_polling_booths_bronze_df.count()
)

written_source_count = (
    written_polling_booths_df
    .select("source_file_path")
    .distinct()
    .count()
)

successful_source_count = (
    written_polling_booths_df
    .filter(F.col("parse_status") == "SUCCESS")
    .select("source_file_path")
    .distinct()
    .count()
)


print("Expected rows:", expected_row_count)
print("Rows read from Delta:", written_row_count)
print("Source files written:", written_source_count)
print("Sources with successful extraction:", successful_source_count)

assert written_row_count == expected_row_count
assert written_source_count == 3
assert successful_source_count == 3

display(
    written_polling_booths_df
    .groupBy(
        "election_year",
        "parse_status",
        "extraction_method"
    )
    .count()
    .orderBy("election_year")
)

print("Polling-booth Bronze write validation: PASS")